# Post-registration: language-model classifier with a larger answer limit

Not pre-registered. Plan committed before the run: docs/LLM_RERUN_PLAN.md. H10 stays REFUTED.

1. Run the cells in order.
2. Paste your API key into the hidden prompt. It is not saved.
3. About 200 calls, 30-60 minutes.
4. A zip downloads at the end. Then delete the API key in the provider's console.

The model's name is never printed or saved; outputs record only its fingerprint.

In [ ]:
# Cell 1: get the code (records the exact commit used)
import subprocess, os
if not os.path.exists("RWA-Asset-tokenization"):
    subprocess.run(["git", "clone", "-q", "https://github.com/FahdSaif/RWA-Asset-tokenization.git"], check=True)
print(subprocess.run(["git", "-C", "RWA-Asset-tokenization", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)


In [ ]:
# Cell 2: key, model and network wrapper (nothing to edit)
import getpass, hashlib, json, urllib.request, urllib.error
from pathlib import Path

FROZEN_TAG = "model-sha256:7d811269d13fd7df0fd2cb573cbd1b11679a1ae347c179b3daf22b12408604c4"
key = getpass.getpass("API key (hidden): ").strip()
req = urllib.request.Request("https://api.anthropic.com/v1/models?limit=100",
                             headers={"x-api-key": key, "anthropic-version": "2023-06-01"})
try:
    models = [m["id"] for m in json.loads(urllib.request.urlopen(req, timeout=30).read())["data"]]
except urllib.error.HTTPError as e:
    raise SystemExit(f"The API refused the key ({e.code}).") from None
# the same model as the frozen run, found by its fingerprint; no other model is used
model = next((m for m in models if "model-sha256:" + hashlib.sha256(m.encode()).hexdigest() == FROZEN_TAG), None)
if model is None:
    raise SystemExit("stop: the frozen model is not available to this key")
print("using the frozen model:", FROZEN_TAG)

# network wrapper: retry when the service is busy (same as the frozen run; frozen code unchanged)
Path("wrap").mkdir(exist_ok=True)
Path("wrap/sitecustomize.py").write_text('''
import time, urllib.error, urllib.request
_open = urllib.request.urlopen
def _retry(req, *a, **k):
    for i in range(8):
        try:
            return _open(req, *a, **k)
        except urllib.error.HTTPError as e:
            if e.code not in (429, 500, 502, 503, 529) or i == 7:
                raise
        except (TimeoutError, urllib.error.URLError):
            if i == 7:
                raise
        time.sleep(min(60, 5 * 2 ** i))
urllib.request.urlopen = _retry
''')
print("ready")


In [ ]:
# Cell 3: run arm A, arm B and the report (about 30-60 minutes; run once)
import sys
root = Path("RWA-Asset-tokenization/rwa-quantumclock")
env = dict(os.environ, PYTHONPATH=str(Path("wrap").resolve()), ANTHROPIC_API_KEY=key, QAMP_LLM_MODEL=model)
for what in ("A", "B", "report"):
    r = subprocess.run([sys.executable, "tools/llm_rerun.py", what], cwd=root, env=env, capture_output=True, text=True)
    print(r.stdout.replace(model, FROZEN_TAG))
    if r.returncode != 0:
        print(r.stderr[-1500:].replace(model, FROZEN_TAG)); raise SystemExit("run failed")


In [ ]:
# Cell 4: download the results (one zip)
import zipfile
from google.colab import files
out = root / "results/exploratory"
names = ["llm_rerun.json", "llm_rerun.md", "llm_rerun_A.json", "llm_rerun_B.json", "llm_rerun_A_log.jsonl", "llm_rerun_B_log.jsonl"]
with zipfile.ZipFile("llm_rerun_results.zip", "w") as z:
    for n in names:
        z.write(out / n, n)
files.download("llm_rerun_results.zip")
del key
print("Now delete the API key in the provider's console.")
